In [1]:
import os

%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\Chicken-Disease-Classification\\notebooks'

In [2]:
os.chdir('../')

%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\Chicken-Disease-Classification'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    checkpoint_dir: Path
    best_model_path: Path
    early_stopping_patience: int

In [4]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks_config
        create_directories([config.root_dir, config.checkpoint_dir])
        
        return PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            checkpoint_dir=Path(config.checkpoint_dir),
            best_model_path=Path(config.best_model_path),
            early_stopping_patience=config.early_stopping_patience
        ) 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class PrepareCallbacks:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config
        self.best_metric = float('inf')
        self.patience_counter = 0
        
    def save_checkpoint(self, model: nn.Module, optimizer, epoch: int, metric: float):
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metric": metric
        }
        
        path = self.config.checkpoint_dir / f"checkpoint_epoch_{epoch}th.pth"
        torch.save(ckpt, path)
    
    def check_early_stopping(self, metric: float) -> bool:
        if metric < self.best_metric:
            self.patience_counter = 0
        else:
            self.patience_counter += 1
        
        return self.patience_counter >= self.config.early_stopping_patience

    def save_best_model(self, model: nn.Module, metric: float) -> bool:
        if metric < self.best_metric:
            self.best_metric = metric
            torch.save(model.state_dict(), self.config.best_model_path)
            return True  # signal: best model updated
        
        return False

In [6]:
import sys
from chicken_disease_classification.exception.exception import CustomException

try:
    config = ConfigManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallbacks(config=prepare_callbacks_config)
    print("PrepareCallbacks initialized successfully")
    print(f"Checkpoint dir: {prepare_callbacks_config.checkpoint_dir}")
    print(f"Best model path: {prepare_callbacks_config.best_model_path}")
    print(f"Early stopping patience: {prepare_callbacks_config.early_stopping_patience}")

except Exception as e:
    raise CustomException(e, sys)

[2026-03-31 12:37:30,405: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-31 12:37:30,417: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-31 12:37:30,426: INFO: common: created directory at: artifacts]
[2026-03-31 12:37:30,434: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-03-31 12:37:30,444: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoints]
PrepareCallbacks initialized successfully
Checkpoint dir: artifacts\prepare_callbacks\checkpoints
Best model path: artifacts\prepare_callbacks\best_model.pth
Early stopping patience: 5
